In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used

# Energy emission peaks in MeV
Cs137_MeV = 0.662
Na22_MeV = [0.511, 1.275, 1.786]
Co60_MeV = [1.173, 1.332, 2,505]
Am241_MeV = 0.0595
# Background emission
K40_MeV = 1.460
Tl208_MEV = 2.614

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kViolet, ROOT.kYellow, ROOT.kSpring, ROOT.kCyan,]

In [2]:
files = sorted((project_root/'data/260427').glob('SUBT_*'))
acqs = [SiphraAcquisition(file) for file in files]
#print(acqs)
bg_acq = acqs[12]
na_acq = acqs[11]
cs_acq = acqs[13]
co_acq = acqs[14]

long_acqs = acqs[11:15]
print(long_acqs)

bg_acqs = [bg_acq, bg_acq, bg_acq]
src_acqs = [na_acq, cs_acq, co_acq]


[SIPHRAAcquisition(File: 'SUBT_12_SIPHRA16Test_1--100_1--3_triggerOnAll_Na22'), SIPHRAAcquisition(File: 'SUBT_13_SIPHRA16Test_1--100_1--3_triggerOnAll_BG'), SIPHRAAcquisition(File: 'SUBT_14_SIPHRA16Test_1--100_1--3_triggerOnAll_Cs137'), SIPHRAAcquisition(File: 'SUBT_15_SIPHRA16Test_1--100_1--3_triggerOnAll_Co60')]


In [3]:
# histograms
hists = {}
sources = []
for sgnl, bg in zip(src_acqs, bg_acqs):
    filepath = sgnl.filepath.stem
    src = (MetadataLoader.load(sgnl.metadataFile)).source
    print(src)
    sources.append(src)
    # Signal and Background
    hist_sgnl = ROOT.TH1F(f"{src} Signal", "", BITS12, 0, BITS12)
    hist_bg = ROOT.TH1F(f"{src} Background", "", BITS12, 0 , BITS12)
    hist_sgnl.Fill(sgnl['s']/len(sgnl.active_chs))
    hist_bg.Fill(bg['s']/len(bg.active_chs))
    hist_bg.Scale(sgnl.exposure/bg.exposure)
    # Clean spectrum
    hist_clean = hist_sgnl.Clone(f"{src} Clean")
    hist_clean.Add(hist_bg, -1)
    for hist in [hist_sgnl, hist_bg, hist_clean]:
        # hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC")
        hist.GetXaxis().SetTitle("ADC channel number")
        hist.GetYaxis().SetTitle(r"Normalized counts")
    hists[src] = hist_sgnl
    hists[f"{src}_BG"] = hist_bg
    hists[f"{src}_clean"] = hist_clean
    del hist_sgnl, hist_bg

Na-22
Cs-137
Co-60


In [4]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 1600, 1200)
cv.Divide(2,2)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(4)]

for idx, (src, lg) in enumerate(zip(sources, lgds)):   
    cv.cd(idx+1)
    
    sgnl = hists[src]
    bg = hists[src+'_BG']
    clean = hists[src+'_clean']
    
    lg.AddEntry(sgnl, "Signal")
    lg.AddEntry(bg, "Background")
    lg.AddEntry(clean, "Subtracted")
    sgnl.SetLineColor(colors[0])
    bg.SetLineColor(colors[1])
    clean.SetLineColor(colors[2])
    sgnl.SetTitle(src)
    sgnl.Draw("hist")
    bg.Draw("sames hist")
    clean.Draw("sames hist")
    ROOT.gPad.Update()
    for i, sp in enumerate([sgnl, bg, clean]):
        st = sp.FindObject("stats")
        # print(type(st))
        st.SetY1NDC(Yinit - i*0.08)
        st.SetY2NDC(0.1)
    lg.Draw()
    cv.cd(idx+1).SetLogy()
    cv.Draw()

In [5]:
from analysis import *

#Calibration fit based on the 3 peaks of the Na spectrum

hist = hists['Na-22_clean']
energy_ranges = [(50, 150), (200, 300), (300, 400)]
energies = Na22_MeV

c_fit = calibration_fit(hist, energy_ranges, energies)
print(c_fit)

[0.00516025685678266, -0.051348556624454256]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      277.652
NDf                       =           97
Edm                       =  1.16358e-07
NCalls                    =           63
Constant                  =       1196.2   +/-   7.94457     
Mean                      =      107.943   +/-   0.125186    
Sigma                     =      19.9794   +/-   0.123521     	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      182.187
NDf                       =           97
Edm                       =  1.14058e-05
NCalls                    =           62
Constant                  =      331.867   +/-   4.2191      
Mean                      =      259.724   +/-   0.285473    
Sigma                     =      22.6716   +/-   0.325946     	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2              

In [6]:
#Calibrating Cs137 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, bg_acqs[1], BITS12)
hist_cal_bg_Cs137.Scale(1/bg_acqs[1].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, src_acqs[1], BITS12)
hist_cal_src_Cs137.Scale(1/src_acqs[1].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)


# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1600, 800)
cv1.Divide(2,1)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv1.cd(1)
hist_cal_bg_Cs137.SetLineColor(colors[0])
hist_cal_src_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"Signal ^{137}Cs", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
lg1.Draw()
cv1.cd(1).SetLogy()
#hist_cal_bg.GetYaxis().SetRangeUser(0,1e4)


lg2 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv1.cd(2)
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
hist_cal_clean_Cs137.Draw("hist")
lg2.AddEntry(hist_cal_clean_Cs137, r"^{137}Cs bg subtracted", "l")
lg2.Draw()
cv1.cd(2).SetLogy()

cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(0.5, 0.9)], [Cs137_MeV])
print(res_Cs137)

[0.30768156630457044]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      639.031
NDf                       =           74
Edm                       =  2.56748e-06
NCalls                    =           74
Constant                  =      36.5925   +/-   0.0748676   
Mean                      =     0.629124   +/-   0.000193093 
Sigma                     =    0.0864905   +/-   0.000156283  	 (limited)


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [10]:
#Calibration based on the 3 peaks of the Na-22 spectrum
#hist = hists['Na-22_clean']
#energy_ranges = [(90, 120), (200, 300), (300, 400)]
#energies = Na22_MeV

#c_fit = calibration_fit(hist, energy_ranges, energies)

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Na22 = calibrated_histogram(c_fit, bg_acqs[0], BITS12)
hist_cal_bg_Na22.Scale(1/bg_acqs[0].exposure) 
hist_cal_src_Na22 = calibrated_histogram(c_fit, src_acqs[0], BITS12)
hist_cal_src_Na22.Scale(1/src_acqs[0].exposure) 
hist_cal_clean_Na22 = hist_cal_src_Na22.Clone("Calibrated signal no BG")
hist_cal_clean_Na22.Add(hist_cal_bg_Na22, -1)


# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv2'):
    ROOT.gROOT.FindObject('cv2').Close()
cv2 = ROOT.TCanvas("cv2", "cv2", 1600, 800)
cv2.Divide(2,1)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv2.cd(1)
hist_cal_bg_Na22.SetLineColor(colors[0])
hist_cal_src_Na22.SetLineColor(colors[1])
hist_cal_src_Na22.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Na22.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Na22, "Background", "l")
lg1.AddEntry(hist_cal_src_Na22, r"Signal ^{22}Na", "l")
hist_cal_src_Na22.Draw("hist")
hist_cal_bg_Na22.Draw("sames hist")
lg1.Draw()
cv2.cd(1).SetLogy()
#hist_cal_bg.GetYaxis().SetRangeUser(0,1e4)


lg2 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv2.cd(2)
hist_cal_clean_Na22.SetLineColor(colors[2])
hist_cal_clean_Na22.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_clean_Na22.GetYaxis().SetTitle(r"Count rate [Hz]")
hist_cal_clean_Na22.Draw("hist")
lg2.AddEntry(hist_cal_clean_Na22, r"^{22}Na bg subtracted", "l")
lg2.Draw()
cv2.cd(2).SetLogy()

cv2.Draw()

#Calculate energy resolution
res_Na22 = energy_resolution(hist_cal_clean_Na22, [(0.3, 0.7), (1.1, 1.5), (1.5 , 2.1)],  Na22_MeV)
print(res_Na22)



[0.46571916734405794, 0.2023269716238369, 0.16820620509989528]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      172.153
NDf                       =           75
Edm                       =  1.48064e-07
NCalls                    =           72
Constant                  =        3.743   +/-   0.0252034   
Mean                      =     0.507254   +/-   0.000655602 
Sigma                     =     0.101054   +/-   0.000678151  	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      78.8819
NDf                       =           75
Edm                       =  4.66429e-07
NCalls                    =           71
Constant                  =      1.05945   +/-   0.0134102   
Mean                      =      1.29219   +/-   0.00143005  
Sigma                     =      0.10954   +/-   0.00161      	 (limited)
****************************************
Minimizer is Minuit2 / Migrad


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [9]:
#Calibrating Co60 histograms based on Na22 calibration fit
hist_cal_bg_Co60 = calibrated_histogram(c_fit, bg_acqs[2], BITS12)
hist_cal_bg_Co60.Scale(1/bg_acqs[2].exposure) 
hist_cal_src_Co60 = calibrated_histogram(c_fit, src_acqs[2], BITS12)
hist_cal_src_Co60.Scale(1/src_acqs[2].exposure) 
hist_cal_clean_Co60 = hist_cal_src_Co60.Clone("Calibrated signal no BG")
hist_cal_clean_Co60.Add(hist_cal_bg_Co60, -1)


# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1600, 800)
cv1.Divide(2,1)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv1.cd(1)
hist_cal_bg_Co60.SetLineColor(colors[0])
hist_cal_src_Co60.SetLineColor(colors[1])
hist_cal_src_Co60.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Co60.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Co60, "Background", "l")
lg1.AddEntry(hist_cal_src_Co60, r"Signal ^{60}Co", "l")
hist_cal_src_Co60.Draw("hist")
hist_cal_bg_Co60.Draw("sames hist")
lg1.Draw()
cv1.cd(1).SetLogy()
#hist_cal_bg.GetYaxis().SetRangeUser(0,1e4)


lg2 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv1.cd(2)
hist_cal_clean_Co60.SetLineColor(colors[2])
hist_cal_src_Co60.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Co60.GetYaxis().SetTitle(r"Count rate [Hz]")
hist_cal_clean_Co60.Draw("hist")
lg2.AddEntry(hist_cal_clean_Co60, r"^{60}Co bg subtracted", "l")
lg2.Draw()
cv1.cd(2).SetLogy()

cv1.Draw()

#Calculate energy resolution
res_Co60 = energy_resolution(hist_cal_clean_Co60, [(2.2, 2.6)], [Co60_MeV[2]])
print(res_Co60)

[0.1292616292122132]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      99.8044
NDf                       =           75
Edm                       =  4.07387e-07
NCalls                    =           72
Constant                  =     0.109306   +/-   0.00426813  
Mean                      =      2.37183   +/-   0.00472391  
Sigma                     =     0.109776   +/-   0.00509481   	 (limited)


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
